In [ ]:
# needed for accessing dataset for download 
# Authenticate with Hugging Face
token = getpass("Paste your Hugging Face access token: ").strip()

if not token:
    raise ValueError("Token is empty. Please enter a valid token.")

login(token=token, add_to_git_credential=False)

print("Hugging Face login completed.")

In [ ]:

##### code for downloading the data set / if you have the data already no need for this cell 
REPO_ID = "nexar-ai/nexar_collision_prediction"
LIMIT = 1500

OUT_DIR = Path("nexar_data")
VIDEO_DIR = OUT_DIR / "videos"
META_DIR = OUT_DIR / "metadata"

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

dataset = load_dataset(
    REPO_ID,
    split="train",
    streaming=True
).cast_column("video", Video(decode=False))

records = []

for index, example in enumerate(dataset):
    if index >= LIMIT:
        break

    video_id = f"video_{index:06d}"
    video_info = example["video"]
    source_path = video_info.get("path")
    video_bytes = video_info.get("bytes")

    # Download the exact video belonging to this example
    if video_bytes is not None:
        suffix = Path(source_path or "video.mp4").suffix or ".mp4"
        video_path = VIDEO_DIR / f"{video_id}{suffix}"
        video_path.write_bytes(video_bytes)

    elif source_path and source_path.startswith("hf://"):
        # Remove hf://datasets/ and the repo revision from the URL
        # to get the repository-relative path.
        relative_path = source_path.split("/", 4)[-1]
        relative_path = relative_path.split("/", 1)[-1]

        # Expected path: train/negative/01924.mp4
        if "@" in relative_path:
            relative_path = relative_path.split("/", 1)[-1]

        cached_path = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=relative_path
        )

        suffix = Path(relative_path).suffix or ".mp4"
        video_path = VIDEO_DIR / f"{video_id}{suffix}"
        shutil.copy2(cached_path, video_path)

    else:
        raise RuntimeError(
            f"Cannot download video for index {index}: {source_path}"
        )

    # Save metadata from the SAME example
    metadata = {
        key: value
        for key, value in example.items()
        if key != "video"
    }
    metadata.update({
        "dataset_index": index,
        "video_id": video_id,
        "video_file": str(video_path.resolve()),
        "source_video_path": source_path,
    })

    with open(
        META_DIR / f"{video_id}.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)

    records.append(metadata)
    print(f"Downloaded {index + 1}/{LIMIT}: {video_id}, label={metadata.get('label')}")

pd.DataFrame(records).to_csv(OUT_DIR / "metadata.csv", index=False)

print("\nVideos downloaded:", len(records))
print("Output:", OUT_DIR.resolve())